# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type:** Scoring / Ranking

**Why:** We are assigning a priority score to each page so that we can sort them into a ranked review queue for content editors. The goal isn't just to say 'yes' or 'no' (classification), but to say 'which ones first?' so the team spends their limited time on the highest-value opportunities.

In [1]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# If we just classified 'down' vs not, we'd have 16k items to review. We need a continuous score to rank them.
print(f"Total items with 'down' trend: {len(df[df['trend_direction'] == 'down'])}")
print("A classification output of 'yes/no' leaves too many pages for a human to review.")

Total items with 'down' trend: 16262
A classification output of 'yes/no' leaves too many pages for a human to review.


## 2. Target or proxy

**Target/Proxy:** `is_declining_label` (where `trend_direction == 'down'`)

**Source:** This is a *proxy* label derived from a defined rule on the current 90-day window's data. In a stronger capstone setup on the full warehouse, we would use a true observed outcome (e.g., actual traffic drop in the *next* 30 days based on the *prior* 90 days).

In [2]:
# Our proxy target is 'trend_direction' == 'down'
target_counts = df['trend_direction'].value_counts(dropna=False)
print("Target Distribution (Proxy):")
print(target_counts)

# Notice that 'down' is heavily represented. We'll use this proxy for the starter task.


Target Distribution (Proxy):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

**Metric:** Precision@K (e.g., Precision@50)

**Why:** If the content team only has the capacity to review 50 pages a week, we only care about the quality of the top 50 recommendations. Precision@50 measures what fraction of those top 50 pages actually meet our criteria (e.g. truly declining and high-value), which directly maps to the team's real-world success.

In [3]:
# Let's show how Precision@K works conceptually.
# If a simple baseline rule ranked pages by total impressions_90d and we only had capacity for 50:
top_50 = df.nlargest(50, 'impressions_90d')
baseline_precision = (top_50['trend_direction'] == 'down').mean()
print(f"Baseline Precision@50 (sorting just by impressions): {baseline_precision:.2f}")
print("Our ML model needs to beat this baseline Precision@50.")


Baseline Precision@50 (sorting just by impressions): 0.42
Our ML model needs to beat this baseline Precision@50.


## 4. The unit of analysis, as a real dataframe

**Unit:** One row = one content item (page) per client over a trailing 90-day window.

In [4]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Unit of analysis: {len(df)} rows, where 1 row = 1 pseudonymized content item.")
# Show what the target column looks like based on our proxy definition
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
display(df[['content_id', 'client_id', 'impressions_90d', 'trend_direction', 'is_declining_label']].head())

Unit of analysis: 30000 rows, where 1 row = 1 pseudonymized content item.


,content_id,client_id,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,1
3,content_331d6c4de07b,client_19581e27de,11751,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,1


## 5. Why ML beats a fixed rule here

**Why ML:** A fixed rule (like `impressions > 500 AND age > 180`) is brittle. It applies hard cutoffs that miss nuance—a page with 499 impressions is ignored, while a page with 501 is flagged. ML models (like a Random Forest) can weigh multiple signals simultaneously (age, search position, CTR, engagement, and content length) to find hidden patterns and interactions that are too messy to write by hand as if-statements.

In [5]:
# A fixed rule might say 'impressions_90d > 500 AND age > 180 AND trend is down'
rule_based = df[(df['impressions_90d'] > 500) & (df['content_age_days'] > 180) & (df['trend_direction'] == 'down')]
print(f"Fixed rule caught {len(rule_based)} pages.")

# But we miss a lot of nuance. How many high-impression declining pages are younger than 180 days?
missed_nuance = df[(df['impressions_90d'] > 500) & (df['content_age_days'] <= 180) & (df['trend_direction'] == 'down')]
print(f"Missed by fixed rule (younger pages): {len(missed_nuance)} pages.")
print("An ML model can balance these signals dynamically instead of using hard cutoffs.")


Fixed rule caught 5326 pages.
Missed by fixed rule (younger pages): 4630 pages.
An ML model can balance these signals dynamically instead of using hard cutoffs.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.